## Loading The Dataset

In [1]:
# set up hrnet
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np

image_data_dir = "cloth_data_gen/output/images"
keypoint_data_dir = "cloth_data_gen/output/keypoints"

img_arr = []
keypoints_img_arr = []
for img_file in os.listdir(image_data_dir):
    if img_file.endswith('.png'):
        name = img_file.split('.')[0]
        keypoint_file = os.path.join(keypoint_data_dir, name + '.txt')
        image_path = os.path.join(image_data_dir, img_file)
        img = cv2.imread(image_path)
        keypoints = pd.read_csv(keypoint_file)
        pixels_coords = keypoints[['x_pixel', 'y_pixel']].values
        img_arr.append(img)
        kimg = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        for point in pixels_coords:
            kimg[int(point[1]), int(point[0])] = 255
        keypoints_img_arr.append(kimg)
img_arr = np.array(img_arr)
keypoints_img_arr = np.array(keypoints_img_arr)


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class KeypointDataset(Dataset):
    def __init__(self, images, keypoints, transform=None):
        self.images = images.astype(np.float32) / 255.0  # normalize to 0-1
        self.keypoints = keypoints.astype(np.float32) / 255.0
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]  # shape (400, 400, 3)
        kp = self.keypoints[idx]  # shape (4, 2)
        img = np.transpose(img, (2, 0, 1))  # channels first
        sample = {'image': torch.from_numpy(img), 'keypoints': torch.from_numpy(kp)}
        if self.transform:
            sample = self.transform(sample)
        return sample

    
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """(Conv => BN => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class UNetEncoder(nn.Module):
    def __init__(self, in_channels=3, features=[512, 256, 128, 64]):
        super().__init__()
        self.layers = nn.ModuleList()
        prev = in_channels
        for feat in features:
            self.layers.append(DoubleConv(prev, feat))
            prev = feat
        self.pools = nn.ModuleList([nn.MaxPool2d(2) for _ in features])
    def forward(self, x):
        skips = []
        for conv, pool in zip(self.layers, self.pools):
            x = conv(x)
            skips.append(x)
            x = pool(x)
        return x, skips

class UNetDecoder(nn.Module):
    def __init__(self, features=[64, 128, 256, 512], out_channels=1):
        super().__init__()
        rev_feats = features[::-1]
        self.ups = nn.ModuleList()
        self.decoders = nn.ModuleList()
        # First up block: input has 2x last feat channels (from bottleneck)
        self.ups.append(nn.ConvTranspose2d(rev_feats[0]*2, rev_feats[0], 2, stride=2))
        self.decoders.append(DoubleConv(rev_feats[0]*2, rev_feats[0]))
        # Subsequent blocks
        for idx in range(1, len(rev_feats)):
            self.ups.append(nn.ConvTranspose2d(rev_feats[idx-1], rev_feats[idx], 2, stride=2))
            self.decoders.append(DoubleConv(rev_feats[idx]*2, rev_feats[idx]))
        self.final_conv = nn.Conv2d(rev_feats[-1], out_channels, 1)
    def forward(self, x, skips):
        for idx in range(len(self.ups)):
            x = self.ups[idx](x)
            skip = skips[-(idx+1)]
            # Pad if needed to handle rounding issues
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([skip, x], dim=1)
            x = self.decoders[idx](x)
        return self.final_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.encoder = UNetEncoder(in_channels, features)
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.decoder = UNetDecoder(features, out_channels)
    def forward(self, x):
        x_in = x
        enc_out, skips = self.encoder(x_in)
        bottleneck = self.bottleneck(enc_out)
        seg = self.decoder(bottleneck, skips)  # (B, out_channels, H, W)
        seg = torch.squeeze(seg, dim=1)
        return seg



## Training Loop

In [ ]:
load_model = True

In [4]:
import torch.optim as optim

# Suppose images_arr: (n, 400, 400, 3), keypoints_arr: (n, 4, 2)
dataset = KeypointDataset(img_arr, keypoints_img_arr)

# split dataset into train and test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
trainloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(100.0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not load_model:
    model = UNet().to(device)
    compiled_model = torch.compile(model)
    optimizer = optim.Adam(compiled_model.parameters(), lr=1e-3)

    for epoch in range(300):
        compiled_model.train()
        running_loss = 0.0
        for batch in trainloader:
            images = batch['image'].to(device)
            keypoints = batch['keypoints'].to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, keypoints)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        print(f'Epoch {epoch+1}: Loss {running_loss / len(dataset):.4f}')
    # save the model
    torch.save(compiled_model.state_dict(), 'models/keypoint_model.pth')
else:
    model = UNet().to(device)
    compiled_model = torch.compile(model)
    compiled_model.load_state_dict(torch.load('models/keypoint_model.pth', map_location=device))
    compiled_model.eval()

Epoch 1: Loss 0.0380
Epoch 2: Loss 0.0130
Epoch 3: Loss 0.0122
Epoch 4: Loss 0.0117
Epoch 5: Loss 0.0111
Epoch 6: Loss 0.0107
Epoch 7: Loss 0.0099
Epoch 8: Loss 0.0092
Epoch 9: Loss 0.0084
Epoch 10: Loss 0.0076
Epoch 11: Loss 0.0069
Epoch 12: Loss 0.0064
Epoch 13: Loss 0.0058
Epoch 14: Loss 0.0056
Epoch 15: Loss 0.0051
Epoch 16: Loss 0.0049
Epoch 17: Loss 0.0047
Epoch 18: Loss 0.0044
Epoch 19: Loss 0.0042
Epoch 20: Loss 0.0041
Epoch 21: Loss 0.0039
Epoch 22: Loss 0.0038
Epoch 23: Loss 0.0037
Epoch 24: Loss 0.0035
Epoch 25: Loss 0.0034
Epoch 26: Loss 0.0034
Epoch 27: Loss 0.0032
Epoch 28: Loss 0.0032
Epoch 29: Loss 0.0030
Epoch 30: Loss 0.0030
Epoch 31: Loss 0.0031
Epoch 32: Loss 0.0028
Epoch 33: Loss 0.0027
Epoch 34: Loss 0.0028
Epoch 35: Loss 0.0026
Epoch 36: Loss 0.0026
Epoch 37: Loss 0.0026
Epoch 38: Loss 0.0025
Epoch 39: Loss 0.0025
Epoch 40: Loss 0.0024
Epoch 41: Loss 0.0024
Epoch 42: Loss 0.0024
Epoch 43: Loss 0.0023
Epoch 44: Loss 0.0022
Epoch 45: Loss 0.0023
Epoch 46: Loss 0.00

## Model Evaluation

In [24]:
from torchviz import make_dot

In [ ]:
# Evaluate on the validation set
compiled_model.eval()
val_loss = 0.0
with torch.no_grad():
    for batch in testloader:
        images = batch['image'].to(device)
        keypoints = batch['keypoints'].to(device)
        outputs = compiled_model(images)
        # render the predicted keypoints on the image
        for img, kp in zip(images.cpu().numpy(), outputs.cpu().numpy()):
            img = np.transpose(img, (1, 2, 0))
            # Convert RGB to BGR for OpenCV
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            # overlap the keypoints onto the image
            img = (img * 255).astype(np.uint8)
            
            # if at kp location there's a value > 0.5, draw the pixel red
            for i in range(kp.shape[0]):
                for j in range(kp.shape[1]):
                    if kp[i, j] > 0.1:
                        cv2.circle(img, (j,i), 1, (0,0,255), -1) # Red color for keypoints


            cv2.imshow('Predicted Keypoints', img)
            cv2.waitKey(0)

        loss = criterion(outputs, keypoints)
        val_loss += loss.item() * images.size(0)
    print(f'Validation Loss: {val_loss / len(test_dataset):.4f}')


(128, 128) (128, 128, 3)


In [6]:
import torch
import cv2
import numpy as np
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor


# 配置路徑
checkpoint = "../sam2/checkpoints/sam2.1_hiera_large.pt"
model_cfg  = "configs/sam2.1/sam2.1_hiera_l.yaml"

# -*- coding: utf-8 -*-

import os
import cv2

from ultralytics import YOLO

# 載入分割模型
model = YOLO("yolov8m-seg.pt")

predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

def extract_mask_compare(image_path):
    image_name = os.path.basename(image_path)
    # 推論圖片
    results = model(image_path, conf=0.15)

    # 如果想儲存結果圖：
    box = None
    for result in results:
        for obj in result.summary():
            if obj["name"] == "bed":
                result.save(filename= "results/" + image_name.replace('.jpg', "_output1.jpg"))
                box = obj["box"]
    if box != None:
        input_point = np.array([[(box["x1"] + box["x2"])//2, (box["y1"]+box["y2"])//2]])
        input_label = np.array([1])
        # 載入圖像
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"無法載入圖像: {image_path}")

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        predictor.set_image(image_rgb)
        masks, scores, logits = predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=True # 會自動選擇分數最高的
        )
        best_mask_idx = np.argmax(scores)
        best_mask = masks[best_mask_idx]
        best_score = scores[best_mask_idx]

        # save the best mask to file
        mask_filename = image_name.replace('.jpg', "_mask.jpg")
        cv2.imwrite("results/" + mask_filename, (best_mask * 255).astype(np.uint8))

        # cropped image
        image_rgb[best_mask == 0] = 0  # 將遮罩外的區域設為黑色

        # resize the image to 128x128
        resized_image = cv2.resize(image_rgb, (128, 128))
        single_batch_image = torch.from_numpy(resized_image).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        images = single_batch_image.to(device)
        outputs = compiled_model(images)
        # render the predicted keypoints on the image
        for img, kp in zip(images.detach().cpu().numpy(), outputs.detach().cpu().numpy()):
            img = np.transpose(img, (1, 2, 0))
            # Convert RGB to BGR for OpenCV
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            print(kp.shape, img.shape)
            # overlap the keypoints onto the image
            img = (img * 255).astype(np.uint8)
            # if at kp location there's a value > 0.5, draw the pixel red
            for i in range(kp.shape[0]):
                for j in range(kp.shape[1]):
                    if kp[i, j] > 0.1:
                        cv2.circle(img, (j,i), 1, (0,0,255), -1) # Red color for keypoints


            cv2.imshow('Predicted Keypoints', img)
            cv2.waitKey(0)


img_files = os.listdir("bed-images")
for img_file in img_files:
    if img_file.endswith('.jpg'):
        image_path = os.path.join("bed-images", img_file)
        extract_mask_compare(image_path)
        print(f"Processed {img_file}")


image 1/1 /home/itrib40351/Documents/GitHub/bedSheetFoldingRobot/bed-images/bed2.jpg: 480x640 1 bed, 77.3ms
Speed: 1.9ms preprocess, 77.3ms inference, 74.1ms postprocess per image at shape (1, 3, 480, 640)
(128, 128) (128, 128, 3)
Processed bed2.jpg

image 1/1 /home/itrib40351/Documents/GitHub/bedSheetFoldingRobot/bed-images/bedsheets-french-fleur-bedsheet-blue-1.jpg: 640x640 2 potted plants, 1 bed, 5 vases, 21.4ms
Speed: 5.4ms preprocess, 21.4ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)
(128, 128) (128, 128, 3)
Processed bedsheets-french-fleur-bedsheet-blue-1.jpg

image 1/1 /home/itrib40351/Documents/GitHub/bedSheetFoldingRobot/bed-images/bed5.jpg: 640x640 1 chair, 1 potted plant, 1 bed, 1 vase, 31.5ms
Speed: 6.1ms preprocess, 31.5ms inference, 3.5ms postprocess per image at shape (1, 3, 640, 640)
(128, 128) (128, 128, 3)
Processed bed5.jpg

image 1/1 /home/itrib40351/Documents/GitHub/bedSheetFoldingRobot/bed-images/bed1.jpg: 480x640 1 bed, 17.4ms
Speed: 1.5ms 